# RunVault — Interactive Quickstart Notebook

Welcome to **RunVault**! This notebook demonstrates how to:
1. Initialize experiments and track training runs using `runvault`.
2. Log hyperparameter configurations and structured parameters.
3. Stream scalar metrics (loss, accuracy) across training steps.
4. Query and analyze persisted metadata from the local SQLite store.
5. Automatically capture errors and failures.

## 1. Import RunVault and Initialize an Experiment

`runvault.init()` is the primary entry point. It creates or retrieves an experiment and starts an active run.

In [1]:
import math
import random

import runvault

print(f"RunVault SDK version: {runvault.__version__}")

RunVault SDK version: 0.1.0


## 2. Execute a Baseline Training Run

Using the `with runvault.init(...) as run:` context manager ensures the run is automatically marked `FINISHED` upon clean exit, or `FAILED` if an exception occurs.

In [2]:
with runvault.init(
    experiment="cifar10-benchmark",
    name="baseline-sgd",
    tags={"optimizer": "sgd", "model": "resnet18"},
    description="Baseline training run on CIFAR-10 with standard SGD",
) as run:
    # Log hyperparameters
    run.log_params(
        {
            "learning_rate": 0.01,
            "batch_size": 64,
            "momentum": 0.9,
            "weight_decay": 1e-4,
        }
    )

    print(f"Active Run ID: {run.id}")
    print(f"Status during execution: {run.status.value}")

    # Simulate training loop across 10 epochs
    for epoch in range(10):
        train_loss = 1.8 / (1.0 + 0.3 * epoch) + random.uniform(-0.02, 0.02)
        accuracy = 0.45 + 0.45 * (1.0 - math.exp(-0.25 * epoch))

        run.log_metrics(
            {
                "train_loss": round(train_loss, 4),
                "accuracy": round(accuracy, 4),
            },
            step=epoch,
        )

print(f"Run ended. Final status: {run.status.value}")

Active Run ID: 6b764fff-df0c-4bf4-8f12-35e90d150c06
Status during execution: RUNNING
Run ended. Final status: FINISHED


## 3. Execute a Second Run with Tuned Hyperparameters

In [3]:
with runvault.init(
    experiment="cifar10-benchmark",
    name="tuned-adamw",
    tags={"optimizer": "adamw", "model": "resnet18"},
) as run_tuned:
    run_tuned.log_params(
        {
            "learning_rate": 0.001,
            "batch_size": 128,
            "weight_decay": 0.01,
        }
    )

    for epoch in range(10):
        train_loss = 1.5 / (1.0 + 0.45 * epoch) + random.uniform(-0.01, 0.01)
        accuracy = 0.52 + 0.42 * (1.0 - math.exp(-0.35 * epoch))

        run_tuned.log_metrics(
            {
                "train_loss": round(train_loss, 4),
                "accuracy": round(accuracy, 4),
            },
            step=epoch,
        )

print(f"Tuned run ended. Final status: {run_tuned.status.value}")

Tuned run ended. Final status: FINISHED


## 4. Query Previous Experiments & Runs from SQLite

RunVault stores all metadata in local SQLite (`.runvault/metadata.db`). You can retrieve historical runs, parameters, and metric curves anytime.

In [4]:
exp = runvault.get_experiment_by_name("cifar10-benchmark")
assert exp is not None

print(f"Experiment: {exp.name}")
print(f"Created: {exp.created_at.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("-" * 60)

for r in exp.list_runs():
    params = runvault.get_params(r.id)
    tags = runvault.get_tags("run", r.id)
    print(f"Run: {r.name:<15} | Status: {r.status.value:<9} | Duration: {r.duration_seconds:.2f}s")
    print(f"  Params: {params}")
    print(f"  Tags:   {tags}")
    print()

Experiment: cifar10-benchmark
Created: 2026-11-15 20:13:46 UTC
------------------------------------------------------------
Run: tuned-adamw     | Status: FINISHED  | Duration: 0.00s
  Params: {'batch_size': 128, 'learning_rate': 0.001, 'weight_decay': 0.01}
  Tags:   {'model': 'resnet18', 'optimizer': 'adamw'}

Run: baseline-sgd    | Status: FINISHED  | Duration: 0.01s
  Params: {'batch_size': 64, 'learning_rate': 0.01, 'momentum': 0.9, 'weight_decay': 0.0001}
  Tags:   {'model': 'resnet18', 'optimizer': 'sgd'}



## 5. Inspect Metric Trajectory

In [5]:
history = runvault.get_metric_history(run_tuned.id, metric_name="accuracy")

print(f"Accuracy history for '{run_tuned.name}':")
for point in history:
    ts = point.timestamp.strftime("%H:%M:%S")
    print(f"  Step {point.step:>2}: {point.value:.4f} (at {ts})")

Accuracy history for 'tuned-adamw':
  Step  0: 0.5200 (at 20:13:54)
  Step  1: 0.6440 (at 20:13:54)
  Step  2: 0.7314 (at 20:13:54)
  Step  3: 0.7930 (at 20:13:54)
  Step  4: 0.8364 (at 20:13:54)
  Step  5: 0.8670 (at 20:13:54)
  Step  6: 0.8886 (at 20:13:54)
  Step  7: 0.9038 (at 20:13:54)
  Step  8: 0.9145 (at 20:13:54)
  Step  9: 0.9220 (at 20:13:54)


## 6. Automatic Failure Tracking

If an unhandled exception aborts a training run, RunVault records the failure details and marks the run as `FAILED` while safely propagating the exception.

In [6]:
try:
    with runvault.init("cifar10-benchmark", name="faulty-run") as faulty_run:
        faulty_run.log_param("stage", "data_loading")
        # Simulate a crash
        raise RuntimeError("CUDA out of memory (simulated error)")
except RuntimeError as e:
    print(f"Caught expected exception: {e}")

# Verify that the run recorded the failure in the database
failed_record = runvault.get_run(faulty_run.id)
assert failed_record is not None
print(f"Run Status:     {failed_record.status.value}")
print(f"Error Type:     {failed_record.error_type}")
print(f"Error Message:  {failed_record.error_message}")

Caught expected exception: CUDA out of memory (simulated error)
Run Status:     FAILED
Error Type:     RuntimeError
Error Message:  CUDA out of memory (simulated error)
